## Project Architecture

**Connecticut RSIP CSV**
↓
**Python Ingestion**
`pd.read_csv()`
↓

### 🥉 Bronze Layer

**Raw Project Data**
Original source data preserved with minimal transformation
↓
**SQL Cleaning & Validation**
Data types • Null handling • Deduplication • Standardization
↓

### 🥈 Silver Layer

**Clean Project Records**
Validated, standardized, and analysis-ready data
↓
**SQL Dimensional Modeling**
Star schema • Derived metrics • Business logic
↓

### 🥇 Gold Layer

**Business-Ready Data Model**
Fact tables • Dimension tables • KPI views
↓

### 📊 Power BI

**Interactive Operations Dashboard**
Project trends • Contractor performance • Cost analysis • Geographic insights


In [3]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/Datasets/Residential_Solar_Investment_Program_(RSIP)_Enrollment_20260911.csv")
df.head()

,Entity,Incentive Type,Sub Program,Approved Date,Completed Date,kW STC,Incentive Amount,Total System Cost,Expected Annual Generation (kWh),Contractor,...,System Financing Type,Utility Company,Municipality,Host Customer City,Host Customer Zip Code,County,CensusTract,MSA Vintage AMI Band,Census Tract Vintage SMI Band,Distressed Community Designation
0,CGB,EPBB,EPBB,2012 Mar 06 12:00:00 AM,2012 May 18 12:00:00 AM,1.92,"3,292","13,632","2,186.496",Sunlight Solar Energy,...,Other Homeowner Purchase,Eversource Energy,Vernon,Vernon,6066,Tolland County,530302,100-120,100-120,Not Distressed
1,CGB,EPBB,EPBB,2012 Mar 07 12:00:00 AM,2012 May 17 12:00:00 AM,4.80,"9,774","29,661","5,466.24",Waldo Renewable Electric,...,Other Homeowner Purchase,United Illuminating,New Haven,New Haven,6501,New Haven County,141900,80-100,80-100,Distressed
2,CGB,EPBB,EPBB,2012 Mar 07 12:00:00 AM,2013 Sep 19 12:00:00 AM,3.99,"8,420","17,987.55","4,543.812",Sunlight Solar Energy,...,Other Homeowner Purchase,Eversource Energy,Guilford,guilford,6437,New Haven County,190100,100-120,100-120,Not Distressed
3,CGB,EPBB,EPBB,2012 Mar 07 12:00:00 AM,2013 Sep 19 12:00:00 AM,6.96,"12,616","39,950.4","7,926.048",Sunlight Solar Energy,...,Other Homeowner Purchase,Eversource Energy,Tolland,Tolland,6084,Tolland County,533102,120+,120+,Not Distressed
4,CGB,EPBB,EPBB,2012 Mar 12 12:00:00 AM,2012 Jul 06 12:00:00 AM,4.08,"8,007","23,306","4,646.304",Lenz Electric,...,Other Homeowner Purchase,Eversource Energy,Ridgefield,Ridgefield,6877,Fairfield County,245100,120+,120+,Not Distressed


In [4]:
df.shape

(48354, 22)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48354 entries, 0 to 48353
Data columns (total 22 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   Entity                            48354 non-null  object 
 1   Incentive Type                    48354 non-null  object 
 2   Sub Program                       48354 non-null  object 
 3   Approved Date                     48354 non-null  object 
 4   Completed Date                    48354 non-null  object 
 5   kW STC                            48354 non-null  float64
 6   Incentive Amount                  48354 non-null  object 
 7   Total System Cost                 48354 non-null  object 
 8   Expected Annual Generation (kWh)  48354 non-null  object 
 9   Contractor                        48354 non-null  object 
 10  System Owner                      48354 non-null  object 
 11  Solarize Indicator                48354 non-null  bool   
 12  Syst

In [6]:
df.columns.tolist()

['Entity',
 'Incentive Type',
 'Sub Program',
 'Approved Date',
 'Completed Date',
 'kW STC',
 'Incentive Amount',
 'Total System Cost',
 'Expected Annual Generation (kWh)',
 'Contractor',
 'System Owner',
 'Solarize Indicator',
 'System Financing Type',
 'Utility Company',
 'Municipality',
 'Host Customer City',
 'Host Customer Zip Code',
 'County',
 'CensusTract',
 'MSA Vintage AMI Band',
 'Census Tract Vintage SMI Band',
 'Distressed Community Designation']

In [7]:
df.isna().sum().sort_values(ascending=False)

,0
System Financing Type,1224
Entity,0
Sub Program,0
Approved Date,0
Completed Date,0
Incentive Type,0
kW STC,0
Incentive Amount,0
Expected Annual Generation (kWh),0
Total System Cost,0


In [9]:
df.duplicated().sum() ##no duplicates found##

np.int64(0)

🥉 Bronze Layer — Raw Data Ingestion

The Connecticut Residential Solar Investment Program dataset is ingested from its original CSV source using Pandas.

At this stage, the data is intentionally preserved in its raw form. Cleaning, type conversions, validation, and business transformations will be performed in the Silver layer.

The source dataset contains 48,354 solar project records across 22 fields.

In [10]:
import duckdb

con = duckdb.connect("/content/drive/MyDrive/Datasets/solar_operations.duckdb")

con.execute("CREATE SCHEMA IF NOT EXISTS bronze")

con.register("raw_df", df)

con.execute("""
CREATE OR REPLACE TABLE bronze.rsip_projects AS
SELECT *
FROM raw_df
""")

## 🥈 Silver Layer — Cleaning and Standardization

Raw Bronze data is transformed into a standardized, analysis-ready dataset.

Silver-layer transformations include:

- Standardizing column names
- Converting dates to proper date types
- Converting monetary and generation fields to numeric values
- Restoring leading zeros in ZIP codes
- Standardizing geographic text fields
- Handling missing financing types
- Creating operational metrics
- Validating project costs, capacity, and project dates

In [11]:
con.execute("""
CREATE SCHEMA IF NOT EXISTS silver;

CREATE OR REPLACE TABLE silver.projects AS

WITH cleaned AS (

    SELECT

        TRIM("Entity") AS entity,

        TRIM("Incentive Type") AS incentive_type,

        TRIM("Sub Program") AS sub_program,


        TRY_STRPTIME(
            "Approved Date",
            '%Y %b %d %I:%M:%S %p'
        )::DATE AS approved_date,

        TRY_STRPTIME(
            "Completed Date",
            '%Y %b %d %I:%M:%S %p'
        )::DATE AS completed_date,


        CAST("kW STC" AS DOUBLE)
            AS system_size_kw,


        TRY_CAST(
            REPLACE("Incentive Amount", ',', '')
            AS DECIMAL(14,2)
        ) AS incentive_amount,


        TRY_CAST(
            REPLACE("Total System Cost", ',', '')
            AS DECIMAL(14,2)
        ) AS total_system_cost,


        TRY_CAST(
            REPLACE(
                "Expected Annual Generation (kWh)",
                ',',
                ''
            )
            AS DOUBLE
        ) AS expected_annual_generation_kwh,


        TRIM("Contractor")
            AS contractor,

        TRIM("System Owner")
            AS system_owner,

        CAST("Solarize Indicator" AS BOOLEAN)
            AS solarize_indicator,


        COALESCE(
            NULLIF(
                TRIM("System Financing Type"),
                ''
            ),
            'Unknown'
        ) AS system_financing_type,


        TRIM("Utility Company")
            AS utility_company,


        UPPER(TRIM("Municipality"))
            AS municipality,


        UPPER(TRIM("Host Customer City"))
            AS host_customer_city,


        LPAD(
            CAST("Host Customer Zip Code" AS VARCHAR),
            5,
            '0'
        ) AS host_customer_zip_code,


        TRIM("County")
            AS county,


        LPAD(
            CAST("CensusTract" AS VARCHAR),
            6,
            '0'
        ) AS census_tract,


        TRIM("MSA Vintage AMI Band")
            AS msa_vintage_ami_band,


        TRIM("Census Tract Vintage SMI Band")
            AS census_tract_vintage_smi_band,


        TRIM("Distressed Community Designation")
            AS distressed_community_designation

    FROM bronze.rsip_projects
)

SELECT

    *,

    ROUND(
        total_system_cost /
        NULLIF(system_size_kw * 1000, 0),
        4
    ) AS cost_per_watt,


    ROUND(
        incentive_amount /
        NULLIF(total_system_cost, 0),
        4
    ) AS incentive_share,


    DATE_DIFF(
        'day',
        approved_date,
        completed_date
    ) AS project_duration_days

FROM cleaned
""")

In [12]:
con.execute("""
SELECT *
FROM silver.projects
LIMIT 10
""").df()

,entity,incentive_type,sub_program,approved_date,completed_date,system_size_kw,incentive_amount,total_system_cost,expected_annual_generation_kwh,contractor,...,host_customer_city,host_customer_zip_code,county,census_tract,msa_vintage_ami_band,census_tract_vintage_smi_band,distressed_community_designation,cost_per_watt,incentive_share,project_duration_days
0,CGB,EPBB,EPBB,2012-03-06,2012-05-18,1.92,3292.0,13632.00,2186.496,Sunlight Solar Energy,...,VERNON,06066,Tolland County,530302,100-120,100-120,Not Distressed,7.1000,0.2415,73
1,CGB,EPBB,EPBB,2012-03-07,2012-05-17,4.80,9774.0,29661.00,5466.240,Waldo Renewable Electric,...,NEW HAVEN,06501,New Haven County,141900,80-100,80-100,Distressed,6.1794,0.3295,71
2,CGB,EPBB,EPBB,2012-03-07,2013-09-19,3.99,8420.0,17987.55,4543.812,Sunlight Solar Energy,...,GUILFORD,06437,New Haven County,190100,100-120,100-120,Not Distressed,4.5082,0.4681,561
3,CGB,EPBB,EPBB,2012-03-07,2013-09-19,6.96,12616.0,39950.40,7926.048,Sunlight Solar Energy,...,TOLLAND,06084,Tolland County,533102,120+,120+,Not Distressed,5.7400,0.3158,561
4,CGB,EPBB,EPBB,2012-03-12,2012-07-06,4.08,8007.0,23306.00,4646.304,Lenz Electric,...,RIDGEFIELD,06877,Fairfield County,245100,120+,120+,Not Distressed,5.7123,0.3436,116
5,CGB,EPBB,EPBB,2012-03-19,2012-06-25,4.56,9011.0,22417.20,5192.928,Sunlight Solar Energy,...,MANSFIELD,06250,Tolland County,881100,120+,120+,Not Distressed,4.9161,0.4020,98
6,CGB,EPBB,EPBB,2012-03-22,2012-09-13,5.64,9555.0,31584.00,6422.832,C-TEC Solar,...,BRISTOL,06010,Hartford County,406100,-60,-60,Distressed,5.6000,0.3025,175
7,CGB,EPBB,EPBB,2012-03-28,2012-09-12,10.92,16839.0,62715.00,12435.696,Mystic Solar,...,STONINGTON,06378,New London County,705200,120+,120+,Not Distressed,5.7431,0.2685,168
8,CGB,EPBB,EPBB,2012-03-29,2012-09-13,6.37,12442.0,41455.12,7254.156,GM Industries,...,ENFIELD,06082,Hartford County,481100,80-100,80-100,Distressed,6.5079,0.3001,168
9,CGB,EPBB,EPBB,2012-03-30,2012-06-07,3.29,6709.0,15684.00,3746.652,Shippee Solar and Construction,...,THOMPSON,06277,Windham County,900100,100-120,100-120,Not Distressed,4.7672,0.4278,69


In [13]:
con.execute("""
DESCRIBE silver.projects
""").df()

,column_name,column_type,null,key,default,extra
0,entity,VARCHAR,YES,None,None,None
1,incentive_type,VARCHAR,YES,None,None,None
2,sub_program,VARCHAR,YES,None,None,None
3,approved_date,DATE,YES,None,None,None
4,completed_date,DATE,YES,None,None,None
5,system_size_kw,DOUBLE,YES,None,None,None
6,incentive_amount,"DECIMAL(14,2)",YES,None,None,None
7,total_system_cost,"DECIMAL(14,2)",YES,None,None,None
8,expected_annual_generation_kwh,DOUBLE,YES,None,None,None
9,contractor,VARCHAR,YES,None,None,None


In [14]:
con.execute("""
SELECT COUNT(*) AS invalid_cost_records
FROM silver.projects
WHERE total_system_cost <= 0
""").df()

,invalid_cost_records
0,37


In [15]:
con.execute("""
SELECT
    total_system_cost,
    system_size_kw,
    contractor,
    host_customer_city
FROM silver.projects
WHERE total_system_cost <= 0
""").df()

,total_system_cost,system_size_kw,contractor,host_customer_city
0,0.0,9.880,CT Electrical,MIDDLEBURY
1,0.0,2.660,Clean Energy Finance and Investment Authority ...,THOMASTON
2,0.0,4.200,Sound Solar Systems,ORANGE
3,0.0,9.450,Solatek,EAST HAMPTON
4,0.0,9.870,Solatek,MANCHESTER
5,0.0,9.030,Sundoor Solar,BERLIN
6,0.0,6.588,Giuffrida Electric,MIDDLETOWN
7,0.0,10.500,"Tim Forget Electric, LLC",NEW LONDON
8,0.0,7.740,"Tim Forget Electric, LLC",MARLBOROUGH
9,0.0,2.700,Sound Solar Systems,NEW CANAAN


## 🥇 Gold Layer — Dimensional Modeling

The Gold layer transforms cleaned Silver data into a business-ready
star schema designed for reporting and Power BI.

**Fact table grain:** One row per solar project record.

### Dimension Tables
- `dim_contractor` — solar installation contractors
- `dim_location` — geographic project attributes
- `dim_utility` — electric utility companies
- `dim_date` — calendar attributes for time-based analysis

### Fact Table
- `fact_projects` — project capacity, costs, incentives, expected
  generation, and project duration

The Gold layer separates descriptive attributes from measurable
business metrics to support efficient operational reporting.

In [18]:
con.execute("""
CREATE SCHEMA IF NOT EXISTS gold
""")

In [19]:
con.execute("""
CREATE OR REPLACE TABLE gold.dim_contractor AS

SELECT
    ROW_NUMBER() OVER (
        ORDER BY contractor
    ) AS contractor_key,
    contractor

FROM (
    SELECT DISTINCT contractor
    FROM silver.projects
    WHERE contractor IS NOT NULL
)
""")

In [20]:
con.execute("""
SELECT *
FROM gold.dim_contractor
LIMIT 10
""").df()

,contractor_key,contractor
0,1,1st Light Energy
1,2,31Solar
2,3,AEC Solar
3,4,Adema Technologies dba Gloria Solar (USA) form...
4,5,Aegis Electrical Systems
5,6,"Affordable Solar Works, LLC"
6,7,"Akeena Solar, Inc."
7,8,All Electric Construction and Communication
8,9,AllGreenIT
9,10,Alteris


In [21]:
con.execute("""
CREATE OR REPLACE TABLE gold.dim_location AS

SELECT
    ROW_NUMBER() OVER (
        ORDER BY
            county,
            municipality,
            host_customer_city,
            host_customer_zip_code,
            census_tract
    ) AS location_key,

    county,
    municipality,
    host_customer_city,
    host_customer_zip_code,
    census_tract,
    msa_vintage_ami_band,
    census_tract_vintage_smi_band,
    distressed_community_designation

FROM (
    SELECT DISTINCT
        county,
        municipality,
        host_customer_city,
        host_customer_zip_code,
        census_tract,
        msa_vintage_ami_band,
        census_tract_vintage_smi_band,
        distressed_community_designation

    FROM silver.projects
)
""")

In [22]:
con.execute("""
CREATE OR REPLACE TABLE gold.dim_utility AS

SELECT
    ROW_NUMBER() OVER (
        ORDER BY utility_company
    ) AS utility_key,
    utility_company

FROM (
    SELECT DISTINCT utility_company
    FROM silver.projects
    WHERE utility_company IS NOT NULL
)
""")

In [24]:
con.execute("""
CREATE OR REPLACE TABLE gold.dim_date AS

WITH dates AS (

    SELECT approved_date AS full_date
    FROM silver.projects
    WHERE approved_date IS NOT NULL

    UNION

    SELECT completed_date AS full_date
    FROM silver.projects
    WHERE completed_date IS NOT NULL
)

SELECT
    CAST(
        STRFTIME(full_date, '%Y%m%d')
        AS INTEGER
    ) AS date_key,

    full_date,
    YEAR(full_date) AS year,
    QUARTER(full_date) AS quarter,
    MONTH(full_date) AS month_number,
    MONTHNAME(full_date) AS month_name

FROM dates

ORDER BY full_date
""")

In [25]:
con.execute("""
SELECT *
FROM gold.dim_date
LIMIT 10
""").df()

,date_key,full_date,year,quarter,month_number,month_name
0,20041001,2004-10-01,2004,4,10,October
1,20041216,2004-12-16,2004,4,12,December
2,20041227,2004-12-27,2004,4,12,December
3,20050101,2005-01-01,2005,1,1,January
4,20050105,2005-01-05,2005,1,1,January
5,20050112,2005-01-12,2005,1,1,January
6,20050126,2005-01-26,2005,1,1,January
7,20050209,2005-02-09,2005,1,2,February
8,20050301,2005-03-01,2005,1,3,March
9,20050310,2005-03-10,2005,1,3,March


In [26]:
con.execute("""
CREATE OR REPLACE TABLE gold.fact_projects AS

SELECT

    ROW_NUMBER() OVER (
        ORDER BY
            p.approved_date,
            p.completed_date,
            p.contractor,
            p.host_customer_zip_code,
            p.system_size_kw
    ) AS project_key,

    c.contractor_key,
    l.location_key,
    u.utility_key,

    CAST(
        STRFTIME(p.approved_date, '%Y%m%d')
        AS INTEGER
    ) AS approved_date_key,

    CAST(
        STRFTIME(p.completed_date, '%Y%m%d')
        AS INTEGER
    ) AS completed_date_key,

    -- Project attributes
    p.entity,
    p.incentive_type,
    p.sub_program,
    p.system_owner,
    p.solarize_indicator,
    p.system_financing_type,

    -- Measures
    p.system_size_kw,
    p.incentive_amount,
    p.total_system_cost,
    p.expected_annual_generation_kwh,
    p.cost_per_watt,
    p.incentive_share,
    p.project_duration_days,

    -- Data quality flag
    CASE
        WHEN p.total_system_cost <= 0
            THEN 'Invalid Cost'

        WHEN p.system_size_kw <= 0
            THEN 'Invalid Capacity'

        WHEN p.project_duration_days < 0
            THEN 'Invalid Date Sequence'

        ELSE 'Valid'
    END AS data_quality_status

FROM silver.projects p

LEFT JOIN gold.dim_contractor c
    ON p.contractor = c.contractor

LEFT JOIN gold.dim_utility u
    ON p.utility_company = u.utility_company

LEFT JOIN gold.dim_location l
    ON p.county IS NOT DISTINCT FROM l.county
    AND p.municipality IS NOT DISTINCT FROM l.municipality
    AND p.host_customer_city
        IS NOT DISTINCT FROM l.host_customer_city
    AND p.host_customer_zip_code
        IS NOT DISTINCT FROM l.host_customer_zip_code
    AND p.census_tract
        IS NOT DISTINCT FROM l.census_tract
    AND p.msa_vintage_ami_band
        IS NOT DISTINCT FROM l.msa_vintage_ami_band
    AND p.census_tract_vintage_smi_band
        IS NOT DISTINCT FROM l.census_tract_vintage_smi_band
    AND p.distressed_community_designation
        IS NOT DISTINCT FROM l.distressed_community_designation
""")

In [27]:
con.execute("""
SELECT
    COUNT(*) AS fact_rows,
    COUNT(DISTINCT project_key) AS unique_project_keys,

    SUM(
        CASE WHEN contractor_key IS NULL
        THEN 1 ELSE 0 END
    ) AS missing_contractor_keys,

    SUM(
        CASE WHEN location_key IS NULL
        THEN 1 ELSE 0 END
    ) AS missing_location_keys,

    SUM(
        CASE WHEN utility_key IS NULL
        THEN 1 ELSE 0 END
    ) AS missing_utility_keys

FROM gold.fact_projects
""").df()

,fact_rows,unique_project_keys,missing_contractor_keys,missing_location_keys,missing_utility_keys
0,48354,48354,0.0,0.0,0.0


In [28]:
con.execute("""
SELECT
    data_quality_status,
    COUNT(*) AS project_count
FROM gold.fact_projects
GROUP BY data_quality_status
ORDER BY project_count DESC
""").df()

,data_quality_status,project_count
0,Valid,47954
1,Invalid Date Sequence,363
2,Invalid Cost,37


In [29]:
con.execute("""
SELECT *
FROM gold.fact_projects
LIMIT 10
""").df()

,project_key,contractor_key,location_key,utility_key,approved_date_key,completed_date_key,entity,incentive_type,sub_program,system_owner,solarize_indicator,system_financing_type,system_size_kw,incentive_amount,total_system_cost,expected_annual_generation_kwh,cost_per_watt,incentive_share,project_duration_days,data_quality_status
0,1,97,2189,1,20041001,20050324,CCEF,EPBB,EPBB,Does Not Apply,False,Unknown,3.00,13296.0,31645.00,3416.400,10.5483,0.4202,174,Valid
1,2,129,1992,1,20041216,20050301,CCEF,EPBB,EPBB,Does Not Apply,False,Unknown,5.94,25000.0,43200.00,6764.472,7.2727,0.5787,75,Valid
2,3,97,818,1,20041227,20050209,CCEF,EPBB,EPBB,Does Not Apply,False,Unknown,3.75,16620.0,34413.00,4270.500,9.1768,0.4830,44,Valid
3,4,129,3520,1,20050101,20050126,CCEF,EPBB,EPBB,Does Not Apply,False,Unknown,5.76,25000.0,41472.00,6559.488,7.2000,0.6028,25,Valid
4,5,129,759,1,20050105,20050516,CCEF,EPBB,EPBB,Does Not Apply,False,Unknown,3.84,17052.0,28415.90,4372.992,7.4000,0.6001,131,Valid
5,6,4,110,1,20050112,20050927,CCEF,EPBB,EPBB,Does Not Apply,False,Unknown,4.00,17762.5,31680.00,4555.200,7.9200,0.5607,258,Valid
6,7,129,168,1,20050310,20050413,CCEF,EPBB,EPBB,Does Not Apply,False,Unknown,5.70,25000.0,38899.74,6491.160,6.8245,0.6427,34,Valid
7,8,129,758,1,20050310,20050506,CCEF,EPBB,EPBB,Does Not Apply,False,Unknown,5.76,25000.0,43533.00,6559.488,7.5578,0.5743,57,Valid
8,9,97,882,1,20050314,20050519,CCEF,EPBB,EPBB,Does Not Apply,False,Unknown,2.59,11431.0,25195.00,2949.492,9.7278,0.4537,66,Valid
9,10,149,2012,1,20050314,20050629,CCEF,EPBB,EPBB,Does Not Apply,False,Unknown,2.76,12372.0,23052.00,3143.088,8.3522,0.5367,107,Valid


In [30]:
con.execute("""
SELECT
    SUM(CASE WHEN contractor_key IS NULL THEN 1 ELSE 0 END)
        AS missing_contractor_keys,

    SUM(CASE WHEN location_key IS NULL THEN 1 ELSE 0 END)
        AS missing_location_keys,

    SUM(CASE WHEN utility_key IS NULL THEN 1 ELSE 0 END)
        AS missing_utility_keys

FROM gold.fact_projects
""").df()

,missing_contractor_keys,missing_location_keys,missing_utility_keys
0,0.0,0.0,0.0


In [31]:
con.execute("""
SELECT
    data_quality_status,
    COUNT(*) AS records
FROM gold.fact_projects
GROUP BY data_quality_status
ORDER BY records DESC
""").df()

,data_quality_status,records
0,Valid,47954
1,Invalid Date Sequence,363
2,Invalid Cost,37


In [32]:
con.execute("""
CREATE OR REPLACE VIEW gold.contractor_performance AS

SELECT
    c.contractor,

    COUNT(*) AS total_projects,

    ROUND(
        SUM(f.system_size_kw),
        2
    ) AS total_capacity_kw,

    ROUND(
        AVG(f.total_system_cost),
        2
    ) AS avg_project_cost,

    ROUND(
        AVG(f.cost_per_watt),
        2
    ) AS avg_cost_per_watt,

    ROUND(
        AVG(f.project_duration_days),
        1
    ) AS avg_project_duration_days

FROM gold.fact_projects f

JOIN gold.dim_contractor c
    ON f.contractor_key = c.contractor_key

WHERE f.data_quality_status = 'Valid'

GROUP BY c.contractor
""")

In [33]:
con.execute("""
SELECT *
FROM gold.contractor_performance
ORDER BY total_projects DESC
LIMIT 20
""").df()

,contractor,total_projects,total_capacity_kw,avg_project_cost,avg_cost_per_watt,avg_project_duration_days
0,Trinity Solar,12391,99685.69,28268.23,3.55,104.7
1,SolarCity,7075,52923.83,37023.84,5.02,198.9
2,PosiGen,3620,24297.69,29021.27,4.34,217.1
3,Sunrun,3082,25243.61,18670.91,2.37,75.0
4,SunPower Capital,2817,27860.46,40430.77,4.17,92.0
5,Vivint Solar,2686,22040.02,29579.92,3.62,157.8
6,Momentum Solar,1708,13049.14,23914.85,3.18,106.8
7,Ross Solar,1376,13669.29,40394.59,4.23,209.4
8,C-TEC Solar,1255,10717.91,32149.10,3.85,163.0
9,Sunlight Solar Energy,1150,8746.22,38939.77,5.60,140.5


In [34]:
con.execute("""
CREATE OR REPLACE VIEW gold.monthly_project_performance AS

SELECT
    d.year,
    d.month_number,
    d.month_name,

    COUNT(*) AS projects_completed,

    ROUND(
        SUM(f.system_size_kw),
        2
    ) AS installed_capacity_kw,

    ROUND(
        AVG(f.cost_per_watt),
        2
    ) AS avg_cost_per_watt,

    ROUND(
        SUM(f.total_system_cost),
        2
    ) AS total_project_cost

FROM gold.fact_projects f

JOIN gold.dim_date d
    ON f.completed_date_key = d.date_key

WHERE f.data_quality_status = 'Valid'

GROUP BY
    d.year,
    d.month_number,
    d.month_name
""")

In [47]:
con.execute("""
SELECT *
FROM gold.monthly_project_performance
LIMIT 20
""").df()

,year,month_number,month_name,projects_completed,installed_capacity_kw,avg_cost_per_watt,total_project_cost
0,2005,3,March,2,8.94,8.91,74845.00
1,2005,2,February,1,3.75,9.18,34413.00
2,2005,6,June,1,2.76,8.35,23052.00
3,2005,8,August,4,13.58,8.47,117524.97
4,2005,11,November,2,10.48,8.12,84896.00
5,2006,6,June,8,37.28,8.43,310282.30
6,2006,8,August,11,51.18,8.55,429120.35
7,2006,9,September,8,38.35,8.83,332592.45
8,2006,12,December,9,42.33,9.45,391978.87
9,2007,6,June,10,46.28,8.76,398722.63


## SQL Business Analysis

The Gold dimensional model is used to answer operational and
business questions relevant to renewable-energy project reporting.

The following analysis demonstrates joins, aggregations, CTEs,
conditional logic, and window functions.

In [48]:
con.execute("""
SELECT
    c.contractor,
    COUNT(*) AS project_count,
    ROUND(SUM(f.system_size_kw), 2) AS total_capacity_kw,
    ROUND(AVG(f.cost_per_watt), 2) AS avg_cost_per_watt

FROM gold.fact_projects f

JOIN gold.dim_contractor c
    ON f.contractor_key = c.contractor_key

WHERE f.data_quality_status = 'Valid'

GROUP BY c.contractor

ORDER BY project_count DESC

LIMIT 15
""").df()

,contractor,project_count,total_capacity_kw,avg_cost_per_watt
0,Trinity Solar,12391,99685.69,3.55
1,SolarCity,7075,52923.83,5.02
2,PosiGen,3620,24297.69,4.34
3,Sunrun,3082,25243.61,2.37
4,SunPower Capital,2817,27860.46,4.17
5,Vivint Solar,2686,22040.02,3.62
6,Momentum Solar,1708,13049.14,3.18
7,Ross Solar,1376,13669.29,4.23
8,C-TEC Solar,1255,10717.91,3.85
9,Sunlight Solar Energy,1150,8746.22,5.60


In [50]:
con.execute("""
WITH yearly_projects AS (

    SELECT
        d.year,
        COUNT(*) AS project_count

    FROM gold.fact_projects f

    JOIN gold.dim_date d
        ON f.completed_date_key = d.date_key

    WHERE f.data_quality_status = 'Valid'

    GROUP BY d.year
),

growth AS (

    SELECT
        year,
        project_count,

        LAG(project_count) OVER (
            ORDER BY year
        ) AS previous_year_projects

    FROM yearly_projects
)

SELECT
    *,

    ROUND(
        100.0 *
        (project_count - previous_year_projects)
        / NULLIF(previous_year_projects, 0),
        2
    ) AS yoy_growth_pct

FROM growth

ORDER BY year
""").df()

,year,project_count,previous_year_projects,yoy_growth_pct
0,2005,31,<NA>,NaN
1,2006,89,31,187.10
2,2007,168,89,88.76
3,2008,271,168,61.31
4,2009,476,271,75.65
5,2010,466,476,-2.10
6,2011,387,466,-16.95
7,2012,329,387,-14.99
8,2013,1038,329,215.50
9,2014,1467,1038,41.33
